# Scanner Combined Results
Aggregate results across multiple target scanners (e.g. `ground_truth_access`,
`tool_failure`) into a single tidy dataframe and produce comparison figures.

Each scanner block in the config specifies its own `target_scanner`, `split`,
explicit `scan_ids`, and `validation_files`. The notebook concatenates every
block's scan rows (with merged validation grades) into one frame and drives
every figure from that frame.

To extend: add another `scanners` entry in the config.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path("/home/jeffm/projects/scanner_evaluation")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

from analysis.scan_utils import load_scan_results, load_validations

pd.set_option("display.max_rows", 20)
pd.set_option("display.max_columns", 30)

Run-specific values (which scanners/scan_ids/validation files to include,
threshold, results subdirectory) are loaded from a YAML file under
`analysis/configs/`. Display invariants (model/benchmark aliases, colors) are
kept inline below.

In [ ]:
import yaml

CONFIG_PATH = PROJECT_ROOT / "analysis" / "configs" / "combined_dev_test.yaml"

with open(CONFIG_PATH) as f:
    _cfg = yaml.safe_load(f)

VIOLATION_THRESHOLD: int = _cfg["violation_threshold"]
SCANNER_SPECS: list[dict] = list(_cfg.get("scanners") or [])
if not SCANNER_SPECS:
    raise ValueError("Config must list at least one entry under `scanners`.")

_results_subdir = _cfg.get("results_subdir")
RESULTS_DIR: Path | None = (
    PROJECT_ROOT / "analysis" / "results" / "combined" / _results_subdir
    if _results_subdir
    else None
)
if RESULTS_DIR is not None:
    RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# --- Display invariants (held constant across runs) ---

BENCHMARK_ALIASES = {
    "swe_bench_verified_mini": "swe_bench",
}

MODEL_NAME_ALIASES = {
    "anthropic/claude-sonnet-4-6": "sonnet-4.6",
    "anthropic/claude-opus-4-6": "opus-4.6",
    "openai/gpt-5.4": "gpt-5.4",
    "openai/gpt-5.4-2026-03-05": "gpt-5.4",
    "openai/gpt-5-mini-2025-08-07": "gpt-5-mini",
    "openai/gpt-5-mini": "gpt-5-mini",
}

def _shorten_model(name):
    if name is None:
        return name
    return MODEL_NAME_ALIASES.get(name, name)

GRADE_LEVELS = [0, 1, 2, 3]
SCORE_COLORS = {0: "#4393c3", 1: "#f4d35e", 2: "#d1495b", 3: "#7f0000"}

# Stable color per target_scanner for grouped bar charts.
TARGET_SCANNER_COLORS = {
    "ground_truth_access": "#1f77b4",
    "tool_failure": "#d62728",
}
_FALLBACK_PALETTE = ["#2ca02c", "#9467bd", "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf"]

def _color_for_target(target: str, _idx_lookup: dict = {}) -> str:
    if target in TARGET_SCANNER_COLORS:
        return TARGET_SCANNER_COLORS[target]
    if target not in _idx_lookup:
        _idx_lookup[target] = _FALLBACK_PALETTE[len(_idx_lookup) % len(_FALLBACK_PALETTE)]
    return _idx_lookup[target]


def save_fig(fig, name: str) -> None:
    if RESULTS_DIR is None:
        return
    path = RESULTS_DIR / f"{name}.png"
    fig.savefig(path, bbox_inches="tight", dpi=150)
    print(f"  saved figure → {path}")


def save_table(df: pd.DataFrame, name: str, *, index: bool = False) -> None:
    if RESULTS_DIR is None:
        return
    path = RESULTS_DIR / f"{name}.csv"
    df.to_csv(path, index=index)
    print(f"  saved table  → {path}")


print(f"CONFIG          = {CONFIG_PATH.relative_to(PROJECT_ROOT)}")
print(f"THRESHOLD       = {VIOLATION_THRESHOLD}")
print(f"RESULTS_DIR     = {RESULTS_DIR if RESULTS_DIR else '(not saving outputs)'}")
print(f"SCANNERS        = {len(SCANNER_SPECS)} block(s)")
for s in SCANNER_SPECS:
    sids = s.get("scan_ids") or []
    vfiles = s.get("validation_files")
    print(f"  - {s['target_scanner']} / {s['split']:<5} "
          f"scan_ids={sids if sids else '(all)'} "
          f"validation_files={vfiles if vfiles else '(all)'}")

## Build aggregated dataframe

For each scanner block: load every `scan_id=*` parquet under
`evals/scans/{target_scanner}/{split}/scan-results`, filter to the requested
scan_ids, attach scanner metadata, then merge in the requested validation
CSV(s) from the matching `validation/` directory.

Blocks are concatenated into a single long-form `combined` frame keyed by
`(target_scanner, scan_id, transcript_id)`. Every figure below is drawn from
this frame.

In [ ]:
def _load_validation_long(
    validation_dir: Path,
    requested_files: list[str] | None,
    target_scanner: str,
) -> pd.DataFrame:
    """Return a tidy frame [transcript_id, validation_grade] for one block."""
    if not validation_dir.exists() or not any(validation_dir.glob("*.csv")):
        print(f"  [{target_scanner}] no validation directory at {validation_dir}")
        return pd.DataFrame(columns=["transcript_id", "validation_grade"])

    wide = load_validations(validation_dir, prefix="")
    grade_cols = [c for c in wide.columns if c != "transcript_id"]
    if not grade_cols:
        print(f"  [{target_scanner}] no grade columns in validation files")
        return pd.DataFrame(columns=["transcript_id", "validation_grade"])

    if requested_files is None:
        selected = grade_cols
    else:
        target_stems = [Path(f).stem for f in requested_files]
        missing = [s for s in target_stems if s not in grade_cols]
        if missing:
            raise FileNotFoundError(
                f"validation_files entries not found in {validation_dir}: {missing}. "
                f"Available: {grade_cols}"
            )
        selected = target_stems

    long = (
        wide[["transcript_id"] + selected]
        .melt(id_vars="transcript_id", var_name="source_file", value_name="validation_grade")
    )
    long["validation_grade"] = pd.to_numeric(long["validation_grade"], errors="coerce")
    long = long.dropna(subset=["validation_grade"])

    conflict_ids = (
        long.groupby("transcript_id")["validation_grade"]
        .nunique()
        .pipe(lambda s: s[s > 1].index.tolist())
    )
    if conflict_ids:
        print(f"  [{target_scanner}] ⚠ {len(conflict_ids)} conflicting transcript_id(s) across files (kept first).")

    out = (
        long.drop_duplicates("transcript_id", keep="first")
        [["transcript_id", "validation_grade"]]
        .reset_index(drop=True)
    )
    per_file = long.groupby("source_file")["transcript_id"].nunique().to_dict()
    print(f"  [{target_scanner}] validation: {len(out):,} unique grades across {len(selected)} file(s) "
          f"({per_file})")
    return out


def _load_block(spec: dict) -> pd.DataFrame:
    target_scanner = spec["target_scanner"]
    split = spec["split"]
    requested_scan_ids = list(spec.get("scan_ids") or [])
    requested_scanner_key = spec.get("scanner_key")
    requested_validation = spec.get("validation_files")

    base = PROJECT_ROOT / "evals" / "scans" / target_scanner / split
    scan_dir = base / "scan-results"
    validation_dir = base / "validation"

    print(f"\n[{target_scanner}/{split}] loading scans from {scan_dir.relative_to(PROJECT_ROOT)}")
    raw = load_scan_results(scan_dir)
    raw = raw.copy()
    raw["target_scanner"] = target_scanner
    raw["split"] = split
    raw["scan_id"] = raw["scanner_source"].str.removeprefix("scan_id=")
    raw["eval_file"] = raw["transcript_source_uri"].fillna("").map(
        lambda p: Path(p).name if p else None
    )
    raw["scanner_model"] = raw["scanner_model"].map(_shorten_model)
    raw["eval_generation_model"] = raw["transcript_model"].map(_shorten_model)
    raw["benchmark"] = raw["transcript_task_set"].map(
        lambda b: BENCHMARK_ALIASES.get(b, b)
    )
    raw["scanner_label"] = (
        raw["scanner_model"].fillna("(no model)").astype(str) + " · " + raw["scan_id"].str[:6]
    )

    present_keys = sorted(raw["scanner_key"].dropna().unique().tolist())
    if requested_scanner_key is not None:
        resolved_key = requested_scanner_key
    elif len(present_keys) == 1:
        resolved_key = present_keys[0]
    else:
        raise RuntimeError(
            f"[{target_scanner}] multiple scanner_keys present ({present_keys}); set `scanner_key` in config."
        )
    raw = raw[raw["scanner_key"] == resolved_key].copy()

    if requested_scan_ids:
        before = raw["scan_id"].nunique()
        raw = raw[raw["scan_id"].isin(requested_scan_ids)].copy()
        missing = set(requested_scan_ids) - set(raw["scan_id"].unique())
        if missing:
            print(f"  [{target_scanner}] ⚠ requested scan_ids not found: {sorted(missing)}")
        print(f"  [{target_scanner}] scan_ids: {raw['scan_id'].nunique()} of {before} kept")
    else:
        print(f"  [{target_scanner}] scan_ids: all {raw['scan_id'].nunique()} included")

    validation = _load_validation_long(validation_dir, requested_validation, target_scanner)
    if not validation.empty:
        raw = raw.merge(validation, on="transcript_id", how="left")
    else:
        raw["validation_grade"] = np.nan

    print(f"  [{target_scanner}] rows: {len(raw):,} | with validation grade: {raw['validation_grade'].notna().sum():,}")
    return raw


_block_frames = [_load_block(spec) for spec in SCANNER_SPECS]

KEEP_COLS = [
    "target_scanner", "split", "scanner_key", "scan_id", "scanner_label",
    "scanner_model", "scan_timestamp", "transcript_id", "value", "value_num",
    "benchmark", "eval_generation_model", "eval_file",
    "transcript_score", "transcript_success",
    "validation_grade",
]
combined = pd.concat(
    [df.reindex(columns=KEEP_COLS) for df in _block_frames],
    ignore_index=True,
)

print()
print(f"Combined frame: {len(combined):,} rows")
print(f"  target_scanners : {sorted(combined['target_scanner'].unique())}")
print(f"  scan_ids        : {combined['scan_id'].nunique()}")
print(f"  transcripts     : {combined['transcript_id'].nunique():,}")
print(f"  with validation : {combined['validation_grade'].notna().sum():,}")
display(combined.head(8))

## Per-scan-id overview

One row per `(target_scanner, scan_id, benchmark, eval_generation_model,
eval_file)` with scanned/validated counts and violation rates.

In [ ]:
def _violation_rate(series: pd.Series) -> float:
    nums = pd.to_numeric(series, errors="coerce").dropna()
    if nums.empty:
        return float("nan")
    return float(nums.ge(VIOLATION_THRESHOLD).mean())


def _grade_counts(series: pd.Series) -> dict:
    nums = pd.to_numeric(series, errors="coerce").dropna()
    counts = nums.astype(int).value_counts().sort_index()
    return {f"n_grade_{int(g)}": int(c) for g, c in counts.items()}


group_cols = [
    "target_scanner", "scan_id", "scanner_label", "scanner_model",
    "benchmark", "eval_generation_model", "eval_file",
]
rows = []
for key, group in combined.groupby(group_cols, dropna=False):
    validated = group.dropna(subset=["validation_grade"])
    rows.append({
        **dict(zip(group_cols, key)),
        "n_scanned": int(len(group)),
        "n_validated": int(len(validated)),
        **_grade_counts(group["value_num"]),
        "scanner_violation_rate": _violation_rate(group["value_num"]),
        "validation_violation_rate": (
            _violation_rate(validated["validation_grade"]) if not validated.empty else float("nan")
        ),
    })
overview = pd.DataFrame(rows).sort_values(group_cols).reset_index(drop=True)

grade_cols = sorted(
    [c for c in overview.columns if c.startswith("n_grade_")],
    key=lambda c: int(c.removeprefix("n_grade_")),
)
overview[grade_cols] = overview[grade_cols].fillna(0).astype(int)
overview = overview[[c for c in overview.columns if c not in grade_cols] + grade_cols]

save_table(overview, "overview")

display_ov = overview.copy()
for col in ["scanner_violation_rate", "validation_violation_rate"]:
    display_ov[col] = display_ov[col].map(lambda v: f"{v:.1%}" if pd.notna(v) else "—")
display(display_ov)


## Grade distributions per target scanner

Stacked bars over `(benchmark, eval_generation_model)`. One subplot per
`(target_scanner, scan_id)` so different scanner types can be compared
side-by-side.

In [ ]:
import matplotlib.colors as mcolors

def _scale_color(color, factor):
    rgb = np.array(mcolors.to_rgb(color))
    return tuple(np.clip(rgb * factor, 0, 1))

def _model_shade(model_name: str) -> float:
    s = str(model_name).lower()
    if "gpt-5" in s or "gpt5" in s or "5.4" in s:
        return 0.65
    return 1.0


def _plot_grouped_stacked_bars(ax, ordered_keys, scanner_models, panel_df, all_grades, title):
    n_models = max(len(scanner_models), 1)
    n_keys = len(ordered_keys)
    x = np.arange(n_keys)
    group_width = 0.9
    bar_width = group_width / n_models
    offsets = [(-group_width / 2) + bar_width / 2 + i * bar_width for i in range(n_models)]

    xtick_labels = [f"{split_val}\n{bench}" for split_val, bench in ordered_keys]

    group_n = []
    for split_val, bench in ordered_keys:
        n = panel_df[
            (panel_df["split"] == split_val)
            & (panel_df["benchmark"] == bench)
        ]["value_num"].dropna().shape[0]
        # Per-bar n (same across models when eval_generation_models cover the same set);
        # take the max so the label reflects the bar height denominator.
        per_bar_ns = []
        for model in scanner_models:
            per_bar_ns.append(
                panel_df[
                    (panel_df["split"] == split_val)
                    & (panel_df["benchmark"] == bench)
                    & (panel_df["scanner_model"] == model)
                ]["value_num"].dropna().shape[0]
            )
        group_n.append(max(per_bar_ns) if per_bar_ns else n)

    for m_idx, model in enumerate(scanner_models):
        positions = x + offsets[m_idx]
        shade = _model_shade(model)
        props_by_grade = {g: [] for g in all_grades}
        for split_val, bench in ordered_keys:
            sub = panel_df[
                (panel_df["split"] == split_val)
                & (panel_df["benchmark"] == bench)
                & (panel_df["scanner_model"] == model)
            ]["value_num"].dropna().astype(int)
            total = len(sub)
            for g in all_grades:
                props_by_grade[g].append((sub == g).sum() / total if total else 0)

        bottom = np.zeros(n_keys)
        for grade in reversed(all_grades):
            heights = np.array(props_by_grade[grade])
            color = _scale_color(SCORE_COLORS.get(grade, "#999999"), shade)
            ax.bar(positions, heights, bar_width, bottom=bottom,
                   color=color, edgecolor="white", linewidth=0.4)
            bottom += heights

    for xi, n in zip(x, group_n):
        if n > 0:
            ax.text(xi, 1.005, f"n={n}", ha="center", va="bottom", fontsize=8)

    ax.set_xticks(x)
    ax.set_xticklabels(xtick_labels, rotation=45, ha="right", fontsize=8)
    #ax.tick_params(axis="x", labelbottom=True) #can use this to force x label on all plots for clarity. I like it shared
    ax.set_xlim(-0.5, n_keys - 0.5)
    ax.set_ylim(0, 1.12)
    ax.set_title(title, fontsize=10)

preferred_scanner_order = ["ground_truth_access", "tool_failure", "answer_format"]  # adjust to your scanners
target_scanners = sorted(
    combined["target_scanner"].dropna().unique().tolist(),
    key=lambda s: (
        preferred_scanner_order.index(s) if s in preferred_scanner_order
        else len(preferred_scanner_order)
    ),
)


if not target_scanners:
    print("No scans loaded.")
else:
    split_order = ["dev", "test"]
    available_splits = [s for s in split_order if s in combined["split"].unique()]
    available_splits += [
        s for s in combined["split"].dropna().unique() if s not in split_order
    ]
    ordered_keys = []
    for split_val in available_splits:
        bench = (
            combined[combined["split"] == split_val]
            .dropna(subset=["benchmark"])["benchmark"]
            .drop_duplicates()
            .sort_values()
            .tolist()
        )
        ordered_keys.extend((split_val, b) for b in bench)

    all_grades = sorted(combined["value_num"].dropna().astype(int).unique())
    preferred_order = ["sonnet-4.6","gpt-5.4"]  # adjust to actual scanner_model strings
    all_models = sorted(
        combined["scanner_model"].dropna().unique().tolist(),
        key=lambda m: preferred_order.index(m) if m in preferred_order else len(preferred_order),
    )


    n_panels = len(target_scanners)
    fig, axes = plt.subplots(
        n_panels, 1,
        figsize=(max(4.0, 0.5 * len(ordered_keys) + 2), 2.8 * n_panels),
        sharey=True,
        sharex=True,
        squeeze=False,
    )
    for idx, target in enumerate(target_scanners):
        ax = axes[idx][0]
        panel_df = combined[combined["target_scanner"] == target]
        scanner_models = [
            m for m in all_models
            if m in set(panel_df["scanner_model"].dropna())
        ]
        _plot_grouped_stacked_bars(
            ax, ordered_keys, scanner_models, panel_df, all_grades, title=target,
        )
        ax.set_ylabel("Proportion")

    grade_handles = [
        plt.Rectangle((0, 0), 1, 1, color=SCORE_COLORS.get(g, "#999999"))
        for g in all_grades
    ]

    ref_grade = all_grades[len(all_grades) // 2] if all_grades else 0
    ref_color = SCORE_COLORS.get(ref_grade, "#999999")
    model_handles = [
        plt.Rectangle((0, 0), 1, 1, color=_scale_color(ref_color, _model_shade(m)))
        for m in all_models
    ]
    grade_legend = fig.legend(
        grade_handles, [str(g) for g in all_grades], title="Grade",
        loc="lower center", bbox_to_anchor=(0.3, -0.02),
        ncol=len(all_grades), frameon=False,
    )
    fig.add_artist(grade_legend)

    ref_grade = all_grades[len(all_grades) // 2] if all_grades else 0
    ref_color = SCORE_COLORS.get(ref_grade, "#999999")
    model_handles = [
        plt.Rectangle((0, 0), 1, 1, color=_scale_color(ref_color, _model_shade(m)))
        for m in all_models
    ]
    fig.legend(
        model_handles, all_models, title="Scanner Model (shade)",
        loc="lower center", bbox_to_anchor=(0.75, -0.02),
        ncol=len(all_models), frameon=False,
    )

    fig.tight_layout(rect=[0, 0.05, 1, 1])

    save_fig(fig, "grade_distribution")
    plt.show()


## Confusion matrices vs validation

One pooled 4×4 per `(target_scanner, scan_id)` (over all benchmarks the
scanner covered), restricted to transcripts that have a validation grade.
Title carries threshold metrics (n, qwκ, F1).

In [ ]:
def _confusion_matrix(rows: np.ndarray, cols: np.ndarray, levels: list[int]) -> np.ndarray:
    idx = {g: i for i, g in enumerate(levels)}
    k = len(levels)
    cm = np.zeros((k, k), dtype=int)
    for r, c in zip(rows, cols):
        if r in idx and c in idx:
            cm[idx[r], idx[c]] += 1
    return cm


def _quadratic_weighted_kappa(cm: np.ndarray) -> float:
    n = cm.sum()
    if n == 0:
        return float("nan")
    k = cm.shape[0]
    if k < 2:
        return float("nan")
    weights = (np.arange(k)[:, None] - np.arange(k)[None, :]) ** 2 / (k - 1) ** 2
    observed = cm / n
    row_marg = cm.sum(axis=1) / n
    col_marg = cm.sum(axis=0) / n
    expected = np.outer(row_marg, col_marg)
    denom = (weights * expected).sum()
    if denom == 0:
        return float("nan")
    return 1.0 - (weights * observed).sum() / denom


def _threshold_metrics(target: np.ndarray, prediction: np.ndarray, *, threshold: float):
    tgt = target >= threshold
    pred = prediction >= threshold
    tp = int((pred & tgt).sum())
    tn = int((~pred & ~tgt).sum())
    fp = int((pred & ~tgt).sum())
    fn = int((~pred & tgt).sum())
    n = len(tgt)
    return {
        "n": n,
        "accuracy": (tp + tn) / n if n else float("nan"),
        "sensitivity": tp / (tp + fn) if (tp + fn) else float("nan"),
        "specificity": tn / (tn + fp) if (tn + fp) else float("nan"),
        "precision": tp / (tp + fp) if (tp + fp) else float("nan"),
        "f1": (2 * tp / (2 * tp + fp + fn)) if (2 * tp + fp + fn) else float("nan"),
        "tp": tp, "tn": tn, "fp": fp, "fn": fn,
    }


def _draw_cm(ax, cm: np.ndarray, *, xlabel: str, ylabel: str, title: str):
    ax.imshow(cm, cmap="Blues", vmin=0)
    ax.set_xticks(range(len(GRADE_LEVELS)))
    ax.set_yticks(range(len(GRADE_LEVELS)))
    ax.set_xticklabels(GRADE_LEVELS)
    ax.set_yticklabels(GRADE_LEVELS)
    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    vmax = cm.max() if cm.max() > 0 else 1
    for i in range(len(GRADE_LEVELS)):
        for j in range(len(GRADE_LEVELS)):
            ax.text(j, i, str(cm[i, j]), ha="center", va="center", fontsize=9,
                    color="white" if cm[i, j] > vmax / 2 else "black")
    ax.set_title(title, fontsize=9)


valid = combined.dropna(subset=["value_num", "validation_grade"]).copy()
if valid.empty:
    print("No transcripts with both a scanner grade and a validation grade.")
else:
    valid["scanner_int"] = valid["value_num"].astype(int)
    valid["validation_int"] = valid["validation_grade"].astype(int)

    pooled_keys = (
        valid.sort_values(["target_scanner", "split", "scanner_model", "scan_timestamp"])
        .drop_duplicates(["target_scanner", "split", "scanner_model"])[
            ["target_scanner", "split", "scanner_model"]
        ]
        .to_dict("records")
    )
    n_panels = len(pooled_keys)
    n_cols = min(4, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.0 * n_cols, 5 * n_rows),
        squeeze=False,
    )
    metric_rows = []
    for idx, info in enumerate(pooled_keys):
        ax = axes[idx // n_cols][idx % n_cols]
        cell = valid[
            (valid["target_scanner"] == info["target_scanner"])
            & (valid["split"] == info["split"])
            & (valid["scanner_model"] == info["scanner_model"])
        ]
        panel_label = f"{info['target_scanner']} / {info['split']}\n{info['scanner_model']}"
        if cell.empty:
            ax.axis("off")
            ax.set_title(f"{panel_label}\n(no data)", fontsize=9)
            continue
        cm = _confusion_matrix(
            cell["validation_int"].to_numpy(),
            cell["scanner_int"].to_numpy(),
            GRADE_LEVELS,
        )
        kappa = _quadratic_weighted_kappa(cm)
        metrics = _threshold_metrics(
            cell["validation_int"].to_numpy(),
            cell["scanner_int"].to_numpy(),
            threshold=VIOLATION_THRESHOLD,
        )
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        f1_str = f"{metrics['f1']:.2f}" if not np.isnan(metrics['f1']) else "—"
        _draw_cm(
            ax, cm,
            xlabel="Scanner",
            ylabel="Validation",
            title=f"{panel_label}\nn={metrics['n']}, qwκ={kappa_str}, F1={f1_str}",
        )
        metric_rows.append({
            "target_scanner": info["target_scanner"],
            "split": info["split"],
            "scanner_model": info["scanner_model"],
            "n_scans": cell["scan_id"].nunique(),
            **metrics,
            "quadratic_weighted_kappa": kappa,
    })

    for idx in range(n_panels, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis("off")
    fig.suptitle("Confusion matrices vs validation grade", y=1.0)
    fig.tight_layout()
    save_fig(fig, "confusion_matrices_pooled")
    plt.show()

    pooled_df = pd.DataFrame(metric_rows)
    save_table(pooled_df, "pooled_metrics")
    display_pooled = pooled_df.copy()
    for col in ["accuracy", "sensitivity", "specificity", "precision"]:
        display_pooled[col] = display_pooled[col].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    display_pooled["f1"] = display_pooled["f1"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display_pooled["quadratic_weighted_kappa"] = display_pooled["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display(display_pooled)

## Scanner Model Agreement
The models seem to disagree to different degrees across the different scanners/evals. Some have much more disagreement than others. Quantify this with qwk.

In [ ]:
import itertools

# Per-model grade per transcript (avg across multiple scan_ids for the same model).
try:
    per_model
except NameError:
    per_model = (
        combined.dropna(subset=["value_num"])
        .groupby(["target_scanner", "split", "transcript_id", "scanner_model"], dropna=False)["value_num"]
        .mean()
        .reset_index()
    )

block_keys = (
    per_model[["target_scanner", "split"]]
    .drop_duplicates()
    .sort_values(["target_scanner", "split"])
    .to_dict("records")
)

# Build the full list of panels up front so we can size one figure.
panels = []  # each entry: dict with target, split, model_a, model_b, joint
skipped_blocks = []

for block in block_keys:
    target, split = block["target_scanner"], block["split"]
    block_pm = per_model[
        (per_model["target_scanner"] == target) & (per_model["split"] == split)
    ]
    model_order = sorted(block_pm["scanner_model"].dropna().unique().tolist())
    if len(model_order) < 2:
        skipped_blocks.append((target, split, len(model_order)))
        continue
    pivot = block_pm.pivot_table(
        index="transcript_id",
        columns="scanner_model",
        values="value_num",
        aggfunc="first",
    )
    for model_a, model_b in itertools.combinations(model_order, 2):
        if model_a not in pivot.columns or model_b not in pivot.columns:
            continue
        panels.append({
            "target_scanner": target,
            "split": split,
            "model_a": model_a,
            "model_b": model_b,
            "joint": pivot[[model_a, model_b]].dropna(),
        })

for target, split, n_models in skipped_blocks:
    print(f"Only {n_models} scanner_model loaded for {target} / {split} — skipping pairwise agreement.")

all_pair_rows = []

if panels:
    n_panels = len(panels)
    n_cols = min(3, n_panels)
    n_rows = (n_panels + n_cols - 1) // n_cols
    fig, axes = plt.subplots(
        n_rows, n_cols,
        figsize=(4.4 * n_cols, 4.0 * n_rows),
        squeeze=False,
    )

    for idx, panel in enumerate(panels):
        ax = axes[idx // n_cols][idx % n_cols]
        target = panel["target_scanner"]
        split = panel["split"]
        model_a = panel["model_a"]
        model_b = panel["model_b"]
        joint = panel["joint"]
        block_label = f"{target} / {split}"

        if joint.empty:
            ax.axis("off")
            ax.set_title(f"{block_label}\n{model_a} vs {model_b}\n(no overlap)", fontsize=9)
            continue

        # Floor to int for confusion matrix (per-model values may be non-integer
        # if a model graded the same transcript under multiple scan_ids).
        ga = np.floor(joint[model_a]).astype(int).to_numpy()
        gb = np.floor(joint[model_b]).astype(int).to_numpy()
        cm = _confusion_matrix(ga, gb, GRADE_LEVELS)
        kappa = _quadratic_weighted_kappa(cm)
        n = int(cm.sum())
        agree = int(np.trace(cm))
        kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
        _draw_cm(
            ax, cm,
            xlabel=model_b,
            ylabel=model_a,
            title=f"{block_label}\n{model_a} vs {model_b}\nn={n}, agree={agree/n:.1%}, qwκ={kappa_str}",
        )
        all_pair_rows.append({
            "target_scanner": target,
            "split": split,
            "model_a": model_a,
            "model_b": model_b,
            "n_paired": n,
            "agree_rate": agree / n if n else float("nan"),
            "mean_abs_diff": float(np.abs(ga - gb).mean()) if n else float("nan"),
            "quadratic_weighted_kappa": kappa,
        })

    for idx in range(n_panels, n_rows * n_cols):
        axes[idx // n_cols][idx % n_cols].axis("off")

    fig.suptitle("Scanner-vs-scanner agreement (across scanner_model)", y=.99)
    fig.tight_layout()
    save_fig(fig, "scanner_agreement_all_blocks")
    plt.show()

if all_pair_rows:
    pair_df = pd.DataFrame(all_pair_rows).sort_values(
        ["target_scanner", "split", "model_a", "model_b"]
    ).reset_index(drop=True)
    save_table(pair_df, "scanner_agreement_all_blocks")

    display_pair = pair_df.copy()
    display_pair["agree_rate"] = display_pair["agree_rate"].map(
        lambda v: f"{v:.1%}" if pd.notna(v) else "—"
    )
    display_pair["mean_abs_diff"] = display_pair["mean_abs_diff"].map(
        lambda v: f"{v:.2f}" if pd.notna(v) else "—"
    )
    display_pair["quadratic_weighted_kappa"] = display_pair["quadratic_weighted_kappa"].map(
        lambda v: f"{v:.3f}" if pd.notna(v) else "—"
    )
    display(display_pair)


## Performance with Combined Scanner
From the dev set there is some evidence that the cases where scanners agree are more accurate than cases where they disagree. Implementing that here to compare against the post validation grades to see if there is higher accuracy.

In [ ]:
# Composite scanner analysis — combine across scanner_model within each
# (target_scanner, split) group. Mirrors the per-scan_id composite logic but
# treats each scanner_model as one "voter".

# One per-model grade per transcript: if a model graded a transcript under
# multiple scan_ids, average those grades (then later we floor for integer
# composites — same convention as the per-scan_id version).
per_model = (
    combined.dropna(subset=["value_num"])
    .groupby(["target_scanner", "split", "transcript_id", "scanner_model"], dropna=False)["value_num"]
    .mean()
    .reset_index()
)

# Validation grades are per (target_scanner, split, transcript_id) — already
# merged onto every row in `combined`, so dedupe to one row per transcript.
val_per_block = (
    combined.dropna(subset=["validation_grade"])
    [["target_scanner", "split", "transcript_id", "validation_grade"]]
    .drop_duplicates(["target_scanner", "split", "transcript_id"])
)

block_keys = (
    per_model[["target_scanner", "split"]]
    .drop_duplicates()
    .sort_values(["target_scanner", "split"])
    .to_dict("records")
)

all_metrics_rows = []
all_bucket_rows = []

for block in block_keys:
    target = block["target_scanner"]
    split = block["split"]
    block_label = f"{target} / {split}"
    display(Markdown(f"## {block_label} — Composite scanners (across scanner_model)"))

    block_pm = per_model[
        (per_model["target_scanner"] == target) & (per_model["split"] == split)
    ]
    model_order = sorted(block_pm["scanner_model"].dropna().unique().tolist())

    if len(model_order) < 2:
        print(f"Only {len(model_order)} scanner_model loaded for {block_label} — skipping.")
        continue

    pivot = block_pm.pivot_table(
        index="transcript_id",
        columns="scanner_model",
        values="value_num",
        aggfunc="first",
    )

    # Require ≥2 scanner_models for a composite to be meaningful.
    grade_count = pivot.notna().sum(axis=1)
    composite = pivot[grade_count >= 2].copy()
    composite["n_scanner_models"] = grade_count[grade_count >= 2]
    composite["composite_floor_mean"] = (
        np.floor(composite[model_order].mean(axis=1, skipna=True)).astype(int)
    )
    composite["composite_max"] = composite[model_order].max(axis=1, skipna=True).astype(int)

    print(f"Transcripts with grades from ≥2 scanner_models: {len(composite):,}")
    contributor_counts = composite["n_scanner_models"].value_counts().sort_index()
    print("Contributor-count distribution:")
    for n, count in contributor_counts.items():
        print(f"  {int(n)} scanner_models → {int(count)} transcripts")

    block_val = val_per_block[
        (val_per_block["target_scanner"] == target) & (val_per_block["split"] == split)
    ][["transcript_id", "validation_grade"]]
    has_validation = not block_val.empty

    if not has_validation:
        print("No validation grades for this block — skipping confusion matrices.")
        composite_eval = composite
    else:
        composite_eval = composite.merge(
            block_val.set_index("transcript_id"),
            left_index=True, right_index=True, how="left",
        )

    composites = [
        ("floor(mean)", "composite_floor_mean"),
        ("max", "composite_max"),
    ]

    if has_validation:
        scored = composite_eval.dropna(subset=["validation_grade"]).copy()
        scored["validation_int"] = scored["validation_grade"].astype(int)
        if scored.empty:
            print("No transcripts with both ≥2 scanner_model grades and a validation grade.")
        else:
            fig, axes = plt.subplots(
                1, len(composites),
                figsize=(4.6 * len(composites), 4.4),
                squeeze=False,
            )
            metrics_rows = []
            for ax, (name, col) in zip(axes[0], composites):
                tgt = scored["validation_int"].to_numpy()
                pred = scored[col].astype(int).to_numpy()
                cm = _confusion_matrix(tgt, pred, GRADE_LEVELS)
                kappa = _quadratic_weighted_kappa(cm)
                metrics = _threshold_metrics(tgt, pred, threshold=VIOLATION_THRESHOLD)
                kappa_str = f"{kappa:.3f}" if not np.isnan(kappa) else "—"
                _draw_cm(
                    ax, cm,
                    xlabel="Composite",
                    ylabel="Validation",
                    title=f"composite = {name}\nn={metrics['n']}, qwκ={kappa_str}",
                )
                metrics_rows.append({
                    "target_scanner": target, "split": split,
                    "composite": name, **metrics,
                    "quadratic_weighted_kappa": kappa,
                })
            fig.suptitle(
                f"{block_label} — Composite vs Validation Grade "
                f"(≥2 scanner_models per transcript)", y=.98,
            )
            fig.tight_layout()
            save_fig(fig, f"composite_vs_validation__{target}__{split}")
            plt.show()

            metrics_df = pd.DataFrame(metrics_rows)
            all_metrics_rows.extend(metrics_rows)
            display_metrics = metrics_df.drop(columns=["target_scanner", "split"]).copy()
            for col in ["accuracy", "sensitivity", "specificity", "precision"]:
                display_metrics[col] = display_metrics[col].map(
                    lambda v: f"{v:.1%}" if pd.notna(v) else "—"
                )
            display_metrics["f1"] = display_metrics["f1"].map(
                lambda v: f"{v:.3f}" if pd.notna(v) else "—"
            )
            display_metrics["quadratic_weighted_kappa"] = display_metrics["quadratic_weighted_kappa"].map(
                lambda v: f"{v:.3f}" if pd.notna(v) else "—"
            )
            display(Markdown(
                f"### {block_label} — Composite metrics vs validation "
                f"(threshold = {VIOLATION_THRESHOLD})"
            ))
            display(display_metrics)

    # Agreement partition: per transcript, look only at the scanner_models
    # that actually graded it (NaN columns are ignored).
    grade_only = composite[model_order]
    flags = grade_only >= VIOLATION_THRESHOLD          # NaN → False
    flags_neg = grade_only < VIOLATION_THRESHOLD       # NaN → False
    n_pos = flags.sum(axis=1)
    n_neg = flags_neg.sum(axis=1)
    n_total = composite["n_scanner_models"]

    bucket = pd.Series("any-disagree", index=composite.index)
    bucket[(n_pos == n_total) | (n_neg == n_total)] = "all-agree"
    composite["agreement_bucket"] = bucket

    summary_rows = []
    for b, group in composite.groupby("agreement_bucket"):
        base = {
            "target_scanner": target, "split": split,
            "bucket": b, "n_transcripts": int(len(group)),
        }

        if not has_validation:
            summary_rows.append({**base, "composite": "—"})
            continue

        with_val = group.merge(
            block_val.set_index("transcript_id"),
            left_index=True, right_index=True, how="left",
        ).dropna(subset=["validation_grade"])

        if with_val.empty:
            summary_rows.append({
                **base,
                "composite": "—",
                "n_validated": 0,
                "validation_violation_rate": float("nan"),
            })
            continue

        v_int = with_val["validation_grade"].astype(int).to_numpy()
        common = {
            **base,
            "n_validated": int(len(with_val)),
            "validation_violation_rate": float((v_int >= VIOLATION_THRESHOLD).mean()),
        }

        if b == "all-agree":
            pred_int = with_val[composites[0][1]].astype(int).to_numpy()
            metrics = _threshold_metrics(v_int, pred_int, threshold=VIOLATION_THRESHOLD)
            summary_rows.append({**common, "composite": "—", **metrics})
        else:
            for name, col in composites:
                pred_int = with_val[col].astype(int).to_numpy()
                metrics = _threshold_metrics(v_int, pred_int, threshold=VIOLATION_THRESHOLD)
                summary_rows.append({**common, "composite": name, **metrics})

    if not summary_rows:
        print(f"{block_label}: composite frame is empty — no agreement partition to report.")
        continue

    bucket_df = (
        pd.DataFrame(summary_rows)
        .sort_values(["bucket", "composite"])
        .reset_index(drop=True)
    )
    all_bucket_rows.extend(summary_rows)
    save_table(bucket_df, f"agreement_partition__{target}__{split}")

    display_bucket = bucket_df.drop(columns=["target_scanner", "split"]).copy()
    if "validation_violation_rate" in display_bucket.columns:
        display_bucket["validation_violation_rate"] = display_bucket["validation_violation_rate"].map(
            lambda v: f"{v:.1%}" if pd.notna(v) else "—"
        )
    for col in ["accuracy", "sensitivity", "specificity", "precision"]:
        if col in display_bucket.columns:
            display_bucket[col] = display_bucket[col].map(
                lambda v: f"{v:.1%}" if pd.notna(v) else "—"
            )
    if "f1" in display_bucket.columns:
        display_bucket["f1"] = display_bucket["f1"].map(
            lambda v: f"{v:.3f}" if pd.notna(v) else "—"
        )
    display(Markdown(f"### {block_label} — Performance partitioned by full agreement"))
    display(display_bucket)

# Combined cross-block tables (saved once, for cross-block comparison).
if all_metrics_rows:
    save_table(pd.DataFrame(all_metrics_rows), "composite_metrics_all_blocks")
if all_bucket_rows:
    save_table(pd.DataFrame(all_bucket_rows), "agreement_partition_all_blocks")
